[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day4_solution.ipynb)

# Day 4 · 정답 — 딥러닝

클래스 · 텐서 · 학습 루프 · 평가

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`lecture` 와 `practice` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 클래스

### 같이 풀기

수업 중에 같이 푼다.

> **빈칸 문제 1.** `Batch` 에 **최적 온도(890)에서 벗어난 정도**를 돌려주는 `gap()` 메서드를 넣는다.

In [ ]:
class Batch:
    def __init__(self, bid, temp):
        self.bid = bid
        self.temp = temp

    def gap(self):
        return abs(self.temp - 890)

b = Batch('B00115', 898.9)

assert abs(b.gap() - 8.9) < 1e-6, f'실제 {b.gap()}'
print('통과')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 2.** `Kiln` 에 정격 온도를 넘었는지 판정하는 `over(temp)` 를 넣는다.

In [ ]:
class Machine:
    def __init__(self, name):
        self.name = name

class Kiln(Machine):
    def __init__(self, name, rated):
        super().__init__(name)
        self.rated = rated

    def over(self, temp):
        return temp > self.rated

k = Kiln('C', 870)

assert k.over(898) is True and k.over(850) is False, '판정이 틀렸다'
print('통과')

## 2. 텐서

### 같이 풀기

수업 중에 같이 푼다.

> **빈칸 문제 3.** `a` 를 3행 2열 텐서로 바꿔 `b` 에 담는다.

In [ ]:
import torch
a = torch.arange(6, dtype=torch.float32)
b = a.reshape(3, 2)

assert b.shape == (3, 2), f'기대 (3, 2), 실제 {tuple(b.shape)}'
print('통과')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 4.** `w = 2.0` 일 때 `loss = (w - 7) ** 2` 의 기울기를 구해 `g` 에 담는다.

In [ ]:
import torch
w = torch.tensor(2.0, requires_grad=True)
loss = (w - 7) ** 2
loss.backward()
g = w.grad.item()

assert abs(g - (-10.0)) < 1e-6, f'기대 -10.0, 실제 {g}'
print('통과')

## 3. 모델과 학습 루프

### 같이 풀기

수업 중에 같이 푼다.

> **빈칸 문제 5.** 은닉층을 **32개**로 키운 `MLP` 를 만들고, 입력 4건을 통과시킨 출력 모양을 확인한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors()
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 32)
        self.fc2 = nn.Linear(32, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = MLP(X_tr.shape[1])
out = model(X_tr[:4])

assert tuple(out.shape) == (4, 1), f'기대 (4, 1), 실제 {tuple(out.shape)}'
assert model.fc1.out_features == 32, '은닉층이 32여야 한다'
print('통과')

> **빈칸 문제 6.** 학습 루프 다섯 줄의 **순서를 채운다.** 100회 돌린 뒤 손실이 줄었는지 본다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
first = None
for epoch in range(100):
    pred = model(X_tr)
    loss = lossfn(pred, y_tr)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if first is None: first = loss.item()
last = loss.item()

assert last < first, f'손실이 줄어야 한다: {first:.4f} → {last:.4f}'
print(f'통과 — {first:.4f} → {last:.4f}')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 7.** `opt.zero_grad()` 를 **빼면** 어떻게 되는지 본다.
기울기가 쌓여 손실이 제대로 안 줄어드는 것을 확인한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.5)
grads = []
for epoch in range(20):
    loss = lossfn(model(X_tr), y_tr)
    pass   # zero_grad 를 일부러 뺀다
    loss.backward()
    opt.step()
    grads.append(model.fc1.weight.grad.abs().mean().item())

assert grads[-1] > grads[0], f'기울기가 쌓여 커진다: {grads[0]:.4f} → {grads[-1]:.4f}'
print(f'통과 — 기울기가 {grads[0]:.4f} 에서 {grads[-1]:.4f} 로 쌓였다')

### 조별 과제

2~3명이 한 조로 상의하며 푼다.

> **실습문제 1.** 학습하면서 **손실 곡선**을 그린다.
200회 돌리며 매 회 손실을 모아 `losses` 에 담고 그린다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
import matplotlib.pyplot as plt
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for epoch in range(200):
    loss = lossfn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()
assert len(losses) == 200, f'200개여야 한다: {len(losses)}'
assert losses[-1] < losses[0] / 2, f'절반 아래로 떨어져야 한다: {losses[0]:.3f} → {losses[-1]:.3f}'
print('통과')

## 4. 평가

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 8.** 학습한 모델의 **테스트 정확도**를 `acc` 에 담는다.
> `torch.no_grad()` 안에서 계산한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(300):
    loss = lossfn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    pred = (torch.sigmoid(model(X_te)) >= 0.5).float()
    acc = (pred == y_te).float().mean().item()

assert acc > 0.85, f'0.85 는 넘어야 한다: {acc}'
print('통과 — 정확도', round(acc, 3))

### 조별 과제

2~3명이 한 조로 상의하며 푼다.

> **실습문제 2.** **shape 에러를 일부러 내 보고** 메시지를 읽는다.
입력 열 수와 `nn.Linear` 의 첫 인자를 다르게 주면 무슨 말이 나오는지 확인한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors()
wrong = nn.Linear(3, 1)      # 실제 열 수는 X_tr.shape[1] 이다
try:
    wrong(X_tr[:4])
except RuntimeError as e:
    msg = str(e)
    print(msg)
print('실제 열 수:', X_tr.shape[1])
assert 'mat1' in msg or 'shape' in msg.lower(), f'shape 관련 메시지여야 한다: {msg}'
print('통과')

## 5. 종합 문제

### 조별 과제

2~3명이 한 조로 상의하며 푼다.

> **실습문제 3.** **은닉층 크기를 바꿔 가며** 정확도를 비교한다.
8 · 16 · 32 · 64 로 각각 300회 학습해 `results` 딕셔너리에 담고 출력한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors()
def run(hidden):
    torch.manual_seed(42)
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(X_tr.shape[1], hidden)
            self.fc2 = nn.Linear(hidden, 1)
        def forward(self, x):
            return self.fc2(torch.relu(self.fc1(x)))
    m = Net()
    lossfn = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(300):
        loss = lossfn(m(X_tr), y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pred = (torch.sigmoid(m(X_te)) >= 0.5).float()
        return (pred == y_te).float().mean().item()

results = {h: run(h) for h in (8, 16, 32, 64)}
for k, v in results.items():
    print(f'hidden {k:>3}  정확도 {v:.3f}')
assert len(results) == 4, f'4가지여야 한다: {results}'
assert all(v > 0.8 for v in results.values()), f'전부 0.8 은 넘는다: {results}'
print('통과')

> **실습문제 4.** **회귀로 바꿔 본다.** `방전용량` 을 맞히는 신경망을 만든다.
> 마지막 층은 그대로 1개, 손실은 `nn.MSELoss()` 를 쓴다.
> 정답 스케일이 크므로 `y` 도 표준화하면 학습이 훨씬 잘 된다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors(target='방전용량')
mu, sd = y_tr.mean(), y_tr.std()
y_tr_s, y_te_s = (y_tr - mu) / sd, (y_te - mu) / sd

torch.manual_seed(42)
class Reg(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_tr.shape[1], 32)
        self.fc2 = nn.Linear(32, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))
model = Reg()
lossfn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(500):
    loss = lossfn(model(X_tr), y_tr_s)
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    pred = model(X_te) * sd + mu
    rmse = ((pred - y_te) ** 2).mean().sqrt().item()
print('RMSE', round(rmse, 2))
assert rmse < 4.0, f'RMSE 4 미만은 나온다: {rmse}'
print('통과')